In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc ffsim matplotlib numpy pyscf qiskit-addon-sqd
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Stichprobebasierte Quantendiagonalisierung vun emm Chemie-Hamiltonian

*Nutzungsschätzung: under ener Minute uf emm Heron r2 Prozessor (HINWEIS: Des isch bloß e Schätzung. Ihri Laufzeit ka variiere.)*
## Hintergrund
In dem Tutorial zeige mir, wie verrauschti Quantenstichproben nachbearbeitet werde, um e Approximation vum Grundzustand vum Stickstoffmolekül $\text{N}_2$ bi Gleichgewichtsbindungslänge z'finde. Dabei verwende mir de [stichprobebasierte Quantendiagonalisierungsalgorithmus (SQD)](https://arxiv.org/abs/2405.05068) für Stichproben us emm 59-Qubit-Quantenschaltkreis (52 System-Qubits + 7 Ancilla-Qubits). E Qiskit-basierte Implementierung isch verfügbar in de [SQD Qiskit Addons](https://github.com/Qiskit/qiskit-addon-sqd), weiteri Details finde Se in de entsprechende [Dokumentation](/guides/qiskit-addons-sqd) mit emm [einfache Beispiel](/guides/qiskit-addons-sqd-get-started) zum Einstieg.

SQD isch e Technik zum Finde vun Eigenwert und Eigenvektore vun Quantenoperatore, wie em Hamiltonian vun emm Quantensystem, under Verwendung vun Quante- und verteiltem klassischem Computing zämme. Des verteilte klassische Computing wird verwendet, um Stichproben vun emm Quantenprozessor z'verarbeite und en Ziel-Hamiltonian in en durch si aufgespannte Unterraum z'projiziere und z'diagonalisiere. Des ermöglicht SQD, robust gegenüber durch Quantenrausche verfälschti Stichproben z'si und mit große Hamiltonians umz'geh, wie Chemie-Hamiltonians mit Millione vun Wechselwirkungstermen, die jenseits de Reichwite vun exakte Diagonalisierungsmethode liege. En SQD-basierte Workflow hät folgende Schritt:

1. Wähle Se en Schaltkreis-Ansatz und wende Se ihn uf emm Quantencomputer uf en Referenzzustand a (in dem Fall de [Hartree-Fock](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)-Zustand).
2. Sampele Se Bitstrings us em resultierendem Quantenzustand.
3. Führe Se des *selbstkonsistente Konfigurationswiederherstellungsverfahre* uf de Bitstrings us, um d'Approximation vum Grundzustand z'erhalte.

SQD funktioniert bekanntermaßen guet, wenn de Ziel-Eigenzustand dünn besetzt isch: D'Wellenfunktion wird vun ener Menge vun Basiszuständ $\mathcal{S} = {|x\rangle }$ getrage, dere Größe nit exponenziell mit de Problemgröße wachst.

### Quantenchemie
D'Eigenschafte vun Moleküle werde weitgehend durch d'Struktur de Elektronen in ihne bestimmt. Als fermionische Teilche könne Elektronen mit emm mathematische Formalismus namens Zweitquantisierung beschriebe werde. D'Idee isch, dass es e Anzahl vun *Orbitale* gibt, vun dene jedes entweder leer isch oder vun emm Fermion besetzt wird. En System vun $N$ Orbitale wird durch en Satz fermionischer Vernichtungsoperatore ${\hat{a}_p}_{p=1}^N$ beschriebe, die d'fermionische Antikommutatorrelationen erfülle:

$$
\begin{align*}
\hat{a}_p \hat{a}_q + \hat{a}_q \hat{a}_p &= 0, \\
\hat{a}_p \hat{a}_q^\dagger + \hat{a}_q^\dagger \hat{a}_p &= \delta_{pq}.
\end{align*}
$$

De adjungierte Operator $\hat{a}_p^\dagger$ wird Erzeugungsoperator g'nannt.

Bis jetzt hät unseri Darstellung de Spin nit berücksichtigt, de e fundamentale Eigenschaft vun Fermione isch. Bim Berücksichtige vum Spin komme d'Orbitale in Paare vor, d'*Raumorbitale* g'nannt werde. Jedes Raumorbital besteht us zwoi *Spinorbitale*, vun dene ois mit "Spin-$\alpha$" und ois mit "Spin-$\beta$" bezeichnet wird. Mir schribe dann $\hat{a}_{p\sigma}$ für de Vernichtungsoperator, de mit em Spinorbital mit Spin $\sigma$ ($\sigma \in {\alpha, \beta}$) im Raumorbital $p$ assoziiert isch. Wenn mir $N$ als Anzahl de Raumorbitale nähme, gibt's insgesamt $2N$ Spinorbitale. De Hilbert-Raum vun dem System wird vun $2^{2N}$ orthonormale Basisvektore aufgespannt, die mit zweiteilige Bitstrings bezeichnet werde: $\lvert z \rangle = \lvert z_\beta z_\alpha \rangle = \lvert z_{\beta, N} \cdots z_{\beta, 1} z_{\alpha, N} \cdots z_{\alpha, 1} \rangle$.

De Hamiltonian vun emm molekulare System ka g'schriebe werde als

$$
\hat{H} = \sum_{ \substack{pr\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac12
\sum_{ \substack{prqs\\\sigma\tau} }
h_{prqs} \,
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma},
$$

wobei d' $h_{pr}$ und d' $h_{prqs}$ komplexe Zahle sind, die molekulare Integrale g'nannt werde und us de Spezifikation vum Molekül mit emm Computerprogramm berechnet werde könne. In dem Tutorial berechne mir d'Integrale mit em Softwarepaket [PySCF](https://pyscf.org/).

Für Details darüber, wie de molekulare Hamiltonian hergeleitet wird, konsultiere Se e Lehrbuch zur Quantenchemie (zum Beispiel *Modern Quantum Chemistry* vun Szabo und Ostlund). Für e übergeordnete Erklärung, wie Quantenchemieprobleme uf Quantencomputer abgebildet werde, schaue Se sich d'Vorlesung [*Mapping Problems to Qubits*](https://youtube.com/watch?v=TyFU6r8uEsE&t=900) vun de Qiskit Global Summer School 2024 a.

### Local Unitary Cluster Jastrow (LUCJ) Ansatz
SQD brucht en Quantenschaltkreis-Ansatz, us dem Stichproben g'zoge werde könne. In dem Tutorial verwende mir de [Local Unitary Cluster Jastrow (LUCJ)](https://pubs.rsc.org/en/content/articlelanding/2023/sc/d3sc02516k) Ansatz wege sinere Kombination us physikalischer Motivation und Hardware-Freundlichkeit.

De LUCJ-Ansatz isch e spezialisierte Form vum allgemeine Unitary Cluster Jastrow (UCJ) Ansatz, de d'Form hät

$$
  \lvert \Psi \rangle = \prod_{\mu=1}^L e^{\hat{K}_\mu} e^{i \hat{J}_\mu} e^{-\hat{K}_\mu} | \Phi_0 \rangle
$$

wobei $\lvert \Phi_0 \rangle$ en Referenzzustand isch, oft de Hartree-Fock-Zustand, und d' $\hat{K}_\mu$ und d' $\hat{J}_\mu$ d'Form hän

$$
\hat{K}_\mu = \sum_{pq, \sigma} K_{pq}^\mu \, \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{q \sigma}
\;,\;
\hat{J}_\mu = \sum_{pq, \sigma\tau} J_{pq,\sigma\tau}^\mu \, \hat{n}_{p \sigma} \hat{n}_{q \tau}
\;,
$$

wobei mir de Teilchenzahloperator definiert hän

$$
\hat{n}_{p \sigma} = \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{p \sigma}.
$$

De Operator $e^{\hat{K}_\mu}$ isch e Orbitalrotation, die mit bekannte Algorithme in linearer Tiefe und under Verwendung linearer Konnektivität implementiert werde ka.

D'Implementierung vum $e^{i \mathcal{J}_k}$ Term vum UCJ-Ansatz erfordert entweder vollständige Konnektivität oder d'Verwendung vun emm fermionischen Swap-Netzwerk, was für verrauschti präfehlertoleranti Quantenprozessore mit begrenzter Konnektivität schwierig isch. D'Idee vum *lokale* UCJ-Ansatz isch, Dünnbesetztheitsbedingunge uf d' $\mathbf{J}^{\alpha\alpha}$- und $\mathbf{J}^{\alpha\beta}$-Matrize ufz'erlege, die's ermögliche, si in konstanter Tiefe uf Qubit-Topologien mit begrenzter Konnektivität z'implementiere. D'Bedingunge werde durch e Liste vun Indizes spezifiziert, die a'zeige, welchi Matrixeinträg im obere Dreieck vun null verschieden si dürfe (da d'Matrize symmetrisch sind, muss nur des obere Dreieck spezifiziert werde). Disi Indizes könne als Orbitalpaare interpretiert werde, die mitanand interagiere dürfe.

Betrachte Se als Beispiel e quadratische Gitter-Qubit-Topologie. Mir könne d' $\alpha$- und d' $\beta$-Orbitale in parallele Linien uf em Gitter platziere, wobei Verbindunge zwischen dene Linien "Sprosse" vun ener Leiterform bilde, wie folgt:

![Qubit-Zuordnungsdiagramm für de LUCJ-Ansatz uf emm quadratische Gitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/baad4e53-5bfd-4cb4-8027-2ec26d27ecdd.avif)

Bi dem Setup sind Orbitale mit demselbe Spin mit ener Linientopologie verbunde, während Orbitale mit unterschiedliche Spin verbunde sind, wenn si dasselbe Raumorbital teile. Des ergibt d'folgende Indexbedingunge für d' $\mathbf{J}$-Matrize:

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, \ldots, N-1}
\end{align*}
$$

Mit andere Wort: Wenn d' $\mathbf{J}$-Matrize nur a dene a'g'gebene Indizes im obere Dreieck vun null verschieden sind, ka de $e^{i \mathcal{J}_k}$ Term uf ener quadratische Topologie ohne Verwendung vun Swap-Gates in konstanter Tiefe implementiert werde. Natürlich macht des Auferlege solche Bedingunge uf de Ansatz ihn weniger ausdrucksstark, sodass möglicherweise mehr Ansatz-Wiederholunge nötig sind.

D'IBM&reg; Hardware hät e Heavy-Hex-Gitter-Qubit-Topologie, in welchem Fall mir e "Zickzack"-Muster verwende könne, des unde dargestellt isch. In dem Muster werde Orbitale mit demselbe Spin uf Qubits mit ener Linientopologie abgebildet (rote und blaue Kreise), und e Verbindung zwischen Orbitale unterschiedliche Spins isch a jedem 4. Raumorbital vorhande, wobei d'Verbindung durch en Ancilla-Qubit (violetti Kreise) ermöglicht wird. In dem Fall sind d'Indexbedingunge

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, 4, 8, \ldots (p \leq N-1)}
\end{align*}
$$

![Qubit-Zuordnungsdiagramm für de LUCJ-Ansatz uf emm Heavy-Hex-Gitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif)

### Selbstkonsistente Konfigurationswiederherstellung
Des selbstkonsistente Konfigurationswiederherstellungsverfahre isch dazu ausgelegt, so viel Signal wie möglich us verrauschte Quantenstichproben ruszhole. Da de molekulare Hamiltonian d'Teilchenzahl und Spin-Z erhält, isch's sinnvoll, en Schaltkreis-Ansatz z'wähle, de disi Symmetriene au erhält. Wenn er uf de Hartree-Fock-Zustand a'g'wendet wird, hät de resultierend Zustand im rauschfreie Fall e festi Teilchenzahl und Spin-Z. Daher sollte d'Spin-$\alpha$- und d'Spin-$\beta$-Hälften vun jedem us dem Zustand g'sampelte Bitstring dasselbe [Hamming-Gewicht](https://en.wikipedia.org/wiki/Hamming_weight) wie im Hartree-Fock-Zustand hän. Wege em Vorhandensein vun Rausche in aktuelle Quantenprozessore werde manche g'messene Bitstrings disi Eigenschaft verletze. E einfache Form de Postselektion würde disi Bitstrings verwerfe, aber des isch verschwenderisch, weil disi Bitstrings vielleicht no ebbis Signal enthält. Des selbstkonsistente Wiederherstellungsverfahre versucht, en Teil vun dem Signal in de Nachbearbeitung wiederherzustelle. Des Verfahre isch iterativ und brucht als Eingabe e Schätzung de durchschnittliche Besetzunge vun jedem Orbital im Grundzustand, die zuerst us de Rohstichproben berechnet wird. Des Verfahre wird in ener Schleife usgeführt, und jede Iteration hät d'folgende Schritt:

1. Für jeden Bitstring, de d'spezifizierte Symmetriene verletzt, flippe Se si Bits mit emm probabilistischen Verfahre, des dazu ausgelegt isch, de Bitstring näher a d'aktuelle Schätzung de durchschnittliche Orbitalbesetzunge z'bringe, um en neue Bitstring z'erhalte.
2. Sammle Se alli alte und neue Bitstrings, die d'Symmetriene erfülle, und entnehme Se Teilmenge vun ener im Voraus g'wählte feste Größe.
3. Für jede Teilmenge vun Bitstrings projiziere Se de Hamiltonian in de durch d'entsprechende Basisvektore aufgespannte Unterraum (luege Se in de [vorige Abschnitt](#quantum-chemistry) für e Beschreibung vun dene Basisvektore) und berechne Se e Grundzustandsschätzung vum projizierten Hamiltonian uf emm klassische Computer.
4. Aktualisiere Se d'Schätzung de durchschnittliche Orbitalbesetzunge mit de Grundzustandsschätzung mit de niedrigste Energie.

### SQD-Workflow-Diagramm
De SQD-Workflow isch im folgende Diagramm dargestellt:

![Workflow-Diagramm vum SQD-Algorithmus](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/fd7e816f-4e2e-4dd7-a7da-f71afb9ca68d.avif)
## Anforderunge
Stellt Se vor em Beginne vun dem Tutorial sicher, dass Se Folgendes installiert hän:

- Qiskit SDK v1.0 oder höher, mit [Visualisierungs](https://docs.quantum.ibm.com/api/qiskit/visualization)-Unterstützung
- Qiskit Runtime v0.22 oder höher (`pip install qiskit-ibm-runtime`)
- SQD Qiskit Addon v0.11 oder höher (`pip install qiskit-addon-sqd`)
- ffsim v0.0.58 oder höher (`pip install ffsim`)
## Setup

In [1]:
import math

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Schritt 1: Klassische Eingabe uf en Quantenproblem abbilden
In dem Tutorial finde mir e Approximation vum Grundzustand vum Molekül im Gleichgewicht im cc-pVDZ-Basissatz. Zuerst spezifiziere mir des Molekül und si Eigenschafte.

In [2]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -108.464957764796
CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb = 8
nelec = (5, 5)


Before constructing the LUCJ ansatz circuit, we first perform a CCSD calculation in the following code cell. The [$t_1$ and $t_2$ amplitudes](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) from this calculation will be used to initialize the parameters of the ansatz.

In [3]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -108.5933309085008  E_corr = -0.1283731437052354


Vor em Konstruiere vum LUCJ-Ansatz-Schaltkreis führe mir zunächst e CCSD-Berechnung in de folgende Code-Zelle durch. D' [$t_1$- und $t_2$-Amplituden](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) us dere Berechnung werde verwendet, um d'Parameter vum Ansatz z'initialisiere.

In [4]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

### Step 2: Optimize for quantum hardware execution

Next, we optimize the circuit for a target hardware. Typically, this step involves initializing the hardware backend and a pass manager for that backend. However, since the LUCJ ansatz is adapted to the hardware connectivity, we already performed these actions in the previous step. All that is left to do is run the pass manager on the circuit to transpile it to an ISA circuit that can be directly executed on the QPU.

In [5]:
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


Jetzt verwende mir [ffsim](https://github.com/qiskit-community/ffsim), um de Ansatz-Schaltkreis z'erstelle. Da unser Molekül en geschlosseschalige Hartree-Fock-Zustand hät, verwende mir d'spin-balancierte Variante vum UCJ-Ansatz, [UCJOpSpinBalanced](https://qiskit-community.github.io/ffsim/api/ffsim.html#ffsim.UCJOpSpinBalanced). Mir übergebe Wechselwirkungspaare, die für e Heavy-Hex-Gitter-Qubit-Topologie geeignet sind (luege Se in de [Hintergrundabschnitt zum LUCJ-Ansatz](#local-unitary-cluster-jastrow-lucj-ansatz)). Mir setze `optimize=True` in de `from_t_amplitudes`-Methode, um d'"komprimierte" Doppelfaktorisierung de $t_2$-Amplituden z'ermögliche (luege Se [The local unitary cluster Jastrow (LUCJ) ansatz](https://qiskit-community.github.io/ffsim/explanations/lucj.html#Parameter-initialization-from-CCSD) in de ffsim-Dokumentation für Details).

In [6]:
rng = np.random.default_rng()
sampler = StatevectorSampler(seed=rng)
job = sampler.run([isa_circuit], shots=100_000)

In [7]:
primitive_result = job.result()
pub_result = primitive_result[0]

### Step 4: Post-process and return result in desired classical format

A useful metric to judge the quality of the QPU output is the number of valid configurations returned. A valid configuration has the correct particle number and spin Z, which means that the right half of the bitstring has Hamming weight equal to the number of spin-up electrons, and the left half has Hamming weight equal to the number of spin-down electrons. The following cell computes the fraction of sampled configurations that are valid.

In [8]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


Mir empfehle d'folgende Schritt, um de Ansatz z'optimiere und hardware-kompatibel z'mache.

- Wähle Se physikalische Qubits (`initial_layout`) us de Ziel-Hardware us, die em zuvor beschriebene "Zickzack"-Muster entspreche. Des Anlege vun Qubits in dem Muster führt zu emm effizienten hardware-kompatible Schaltkreis mit weniger Gates. Do füge mir Code i, um d'Auswahl vum "Zickzack"-Muster z'automatisiere, de e Bewertungsheuristik verwendet, um d'mit em ausgewählte Layout verbundene Fehler z'minimiere.
- Generiere Se en gestuften Pass-Manager mit de Funktion [generate_preset_pass_manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) vun Qiskit mit Ihrer Wahl vun `backend` und `initial_layout`.
- Setze Se d' `pre_init`-Stufe vun Ihrem gestuften Pass-Manager uf `ffsim.qiskit.PRE_INIT`. `ffsim.qiskit.PRE_INIT` enthält Qiskit-Transpiler-Pässe, die Gates in Orbitalrotationen zerleget und dann d'Orbitalrotationen zusammenführet, was zu weniger Gates im endgültige Schaltkreis führt.
- Führe Se de Pass-Manager uf Ihrem Schaltkreis us.
<details>
<summary>Code für automatisierte Auswahl vum "Zickzack"-Layout</summary>

</details>

In [9]:
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625


Now, we estimate the ground state energy of the Hamiltonian using the `diagonalize_fermionic_hamiltonian` function. This function performs the self-consistent configuration recovery procedure to iteratively refine the noisy quantum samples to improve the energy estimate. We pass a callback function so that we can save the intermediate results for later analysis. See the [API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) for explanations of the arguments to `diagonalize_fermionic_hamiltonian`.

Here, we use the `initial_occupancies` argument to `diagonalize_fermionic_hamiltonian` to specify the Hartree-Fock configuration as the initial guess for the orbital occupancies in the ground state. This approach is sensible for systems where the ground state has significant support on the Hartree-Fock configuration, but it might not be appropriate in other situations, though more advanced computational methods might yield better initial guesses in those cases. Specifying `initial_occupancies` also allows configuration recovery to run even if no valid configurations were sampled, as may be the case when sampling a large circuit on a noisy QPU. Without this argument, the configuration recovery would fail and raise an error if no valid configurations were provided.

In [10]:
from functools import partial

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

Iteration 1
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Iteration 2
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Final energy: -108.59275573641656
Final energy error: 0.0032316145694579745


#### Visualize the results

The first plot shows that in this simulation we are already within `1 mH` of the exact answer after the first iteration (chemical accuracy is typically accepted to be `1 kcal/mol` $\approx$ `1.6 mH`). This is a small system, though, and because the samples are noiseless, configuration recovery is not needed. On a larger system run on a noisy QPU, multiple configuration recovery iterations might be needed, and the final accuracy might be worse. Generally, the energy can be improved by allowing more configuration recovery iterations or by increasing the number of samples per batch.

The second plot shows the average occupancy of each spatial orbital after the final iteration. We can see that both the spin-up and spin-down electrons occupy the first five orbitals with high probability in our solutions.

In [11]:
# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/caffd888-e89c-4aa9-8bae-4d1bb723b35e-0.avif" alt="Output of the previous code cell" />

## Schritt 3: Usführe mit Qiskit-Primitive {#schritt-3-ausführen-mit-qiskit-primitiven}
Nach de Optimierung vum Schaltkreis für d'Hardware-Ausführung sind mir bereit, ihn uf de Ziel-Hardware usz'führe und Stichproben für d'Grundzustandsenergieabschätzung z'sammle. Da mir nur en Schaltkreis hän, verwende mir de [Job-Ausführungsmodus](/guides/execution-modes) vun Qiskit Runtime und führe unsere Schaltkreis us.

In [12]:
# ------------------------------ Step 1 ------------------------------
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="cc-pvdz",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Store reference energy from SCI calculation performed separately
reference_energy = -109.22802921665716

print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(f"Using backend {backend.name}")

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()


# ------------------------------ Step 2 ------------------------------

isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")


# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_SQD"]
job = sampler.run([isa_circuit], shots=100_000)
primitive_result = job.result()
pub_result = primitive_result[0]


# ------------------------------ Step 4 ------------------------------

bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)
# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

converged SCF energy = -108.929838385609
norb = 26
nelec = (5, 5)
E(CCSD) = -109.2177884185544  E_corr = -0.2879500329450045
Using backend ibm_boston


Removing interaction (24, 24) from the end.
Removing interaction (20, 20) from the end.


Gate counts: OrderedDict({'sx': 7039, 'rz': 6990, 'cz': 1858, 'x': 61, 'measure': 52, 'barrier': 1})
Fraction of sampled configurations that are valid: 0.02124
Expected fraction of valid configurations from uniformly random bitstrings: 9.607888706852918e-07
Iteration 1
	Subsample 0
		Energy: -109.13889134249762
		Subspace dimension: 120409
	Subsample 1
		Energy: -109.11785470455858
		Subspace dimension: 110889
	Subsample 2
		Energy: -109.13234360554011
		Subspace dimension: 130321
Iteration 2
	Subsample 0
		Energy: -109.16392179579177
		Subspace dimension: 223729
	Subsample 1
		Energy: -109.16281938332986
		Subspace dimension: 223729
	Subsample 2
		Energy: -109.16955816711932
		Subspace dimension: 233289
Iteration 3
	Subsample 0
		Energy: -109.17905772999075
		Subspace dimension: 324900
	Subsample 1
		Energy: -109.17532445048462
		Subspace dimension: 357604
	Subsample 2
		Energy: -109.1733168689756
		Subspace dimension: 348100
Iteration 4
	Subsample 0
		Energy: -109.18437778820451
		Su

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/3858949c-a55d-4ff8-a0fc-54fb53e131b5-3.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Scale SQD chemistry workflows with Dice solver](https://qiskit.github.io/qiskit-addon-sqd/how_tos/integrate_dice_solver.html) - a page showing how to use the more efficient Dice software for diagonalization
- [SQD addon API documentation](https://qiskit.github.io/qiskit-addon-sqd/apidocs/qiskit_addon_sqd.fermion.html#qiskit_addon_sqd.fermion.diagonalize_fermionic_hamiltonian) - reference for the `diagonalize_fermionic_hamiltonian` function
- [*Chemistry beyond the scale of exact diagonalization on a quantum-centric supercomputer*](https://www.science.org/doi/10.1126/sciadv.adu9991) - the paper this tutorial is based on
</Admonition>